# 다이캐스팅 공정 데이터 기반 품질 예측 프로젝트

- 다이캐스팅은 용융 금속을 금형에 고압으로 주입하여 정밀한 제품을 생산하는 공정이다.
- 생산 효율을 높이고 불량품을 낮추기 위해 공정 변수 및 센서 데이터를 분석해서 불량 유형을 자동 예측하는 머신러닝 모델을 개발해야함.
- 실시간 공정 데이터와 품질 검사를 연계하여 품질 예측 시스템을 구축하고자 함.

# 비즈니스 문제 정의

### 현재 상황

- 불량품이 발생해도 육안 검사에 의존하여 판정 기준의 주관성과 검사 속도의 한계로 생산성이 저하됨.
- 불량 발생 원인을 추적하기 힘들어 공정 개선 및 문제 해결이 어려움.
- 공정 데이터와 품질 검사를 효과적으로 매핑하지 못하여 실시간 품질 관리 및 재발 방지 대책이 부족한 상황.

### 분석 목표

- 다이캐스팅 공정 과정에서 발생하는 다양한 불량 유형을 자동으로 예측해주는 머신 러닝 모델 개발
- 공정 데이터와 센서 데이터를 면밀히 분석해 불량 여부와의 관계성 파악
- 불량 발생의 주요 원인을 분석해서 공정의 최적화를 지원
- 실시간 품질 예측 체계를 구축하여 조기 경보 시스템 도입 및 불량률 감소

### 비즈니스 산출물

1) 불량 발생의 주요 원인을 분석하고 이를 시각화하여 제시
2) 불량 유형을 자동으로 예측해주는 머신러닝 모델 제시

# 데이터셋 구성

- **공정(Process)데이터**
1) Shot ID : 주조 샷 고유 식별자
2) Injection Speed : 용탕 주입 속도 (m/s)
3) Die Temperature : 금형 온도
4) Casting Pressure : 주조 압력 (bar)
5) Cooling Time: 냉각 시간 (s)

- **센서(Sensor) 데이터**
1) Mold Temp Sensor : 금형 내 센서 온도 (°C)
2) Hydraulic Pressure : 유압 압력 (bar)
3) Vibration Sensor : 진동값 (Hz)
4) Flow Rate Sensor : 유량 (L/min)

- **불량(Defects) 데이터**
1) Defect Type : 발생한 불량 유형 (미성형, 박리, 기공, 평탄, 개재물 등)
2) Defect Status : 양품(0) / 불량(1) 여부

In [1]:
# ============================================================
# 공정(Process) 주요 컬럼 설명
# ============================================================
# • Shot ID                       : 주조 번호 (붕어빵 틀에 반죽을 한 번 붓고 뚜껑을 닫는 행위에 관한 일련번호)
# • Injection Speed               : 주입 속도 (쇳물을 금형에 얼마나 빨리 밀어 넣는가)
# • Die Temperature               : 금형 온도 (금형 틀이 어느정도의 온도인가)
# • Casting Pressure              : 주조 압력 (쇳물을 다 채운 뒤 강하게 누르는 힘)
# • Cooling Time                  : 냉각 시간 (액체 상태인 금속이 고체가 될 떄까지 기다리는 힘)


# ============================================================
# 공정(Process) 컬럼별 분석 관점
# ============================================================
# • Injection Speed               : 주입 속도가 너무 느리면 쇳물이 금형으로 가다가 굳어버리고, 너무 빠르면 사방으로 튀어서 속에 공기 방울(기공)이 생길 수 있다.
# • Die Temperature               : 금형 온도가 너무 낮으면 쇳물이 일찍 굳고 너무 높으면 금형 수명이 줄거나 제품이 달라붙는 현상이 생긴다.
# • Casting Pressure              : 압력이 너무 낮으면 기공이 발생할 수 있고, 너무 높으면 금형 손상 및 제품 변형이 일어날 수 있다.
# • Cooling Time                  : 냉각 시간이 너무 짧으면 탈형 시 변형이 오고, 너무 느리면 사이클 타임이 늘어나 생산성이 떨어진다.


# ============================================================
# 센서(Sensor) 주요 컬럼 설명
# ============================================================
# • Mold Temp Sensor               : 금형 내부 온도 (Die Temperature가 전체의 겉 온도라면, 이거는 틀 안쪽 깊숙한 곳의 실시간 온도이다)
# • Hydraulic Pressure             : 유압 압력 (bar) (쇳물을 밀어내기 위해 기계 팔(실린더)이 쓰는 기름의 힘 혈압과 비슷하다고 볼 수 있음)
# • Vibration Sensor               : 진동값 (Hz) (기계가 작동할 떄의 떨림)
# • Flow Rate Sensor               : 유량 (L/min) (틀을 식히기 위한 냉각수가 흐르는 속도를 나타냄)


# ============================================================
# 센서(Sensor) 컬럼별 분석 관점
# ============================================================
# • Mold Temp Sensor               : 쇳물이 들어올 때 순간 확 올랐다가 식으면서 툭 떨어지는 경향을 보인다
# • Hydraulic Pressure             : 유압 압력은 혈압과 비슷하다고 생각할 수 있다. 유압 압력이 일정해야 쇳물을 쏘아 주는 힘이 균일해진다.
# • Vibration Sensor               : 평소보다 많이 흔들린다면 나사가 풀렸거나, 부품이 마모됐거나, 쇳물이 튀어서 어딘가 걸렸다는 신호일 수 있다.
# • Flow Rate Sensor               : 유량이 적으면 틀이 식지 않아서 제품이 녹아내릴 수 있다.


# ============================================================
# 불량(Defects) 주요 컬럼 설명
# ============================================================
# • Defect Type                   : 발생한 불량 유형
# • Defect Status                 : 양품(0) / 불량(1) 여부


# ============================================================
# 불량(Defects) 유형 설명
# ============================================================
# • 미성형 (Underfill/Short Shot)  : 반죽이 모자라 다 완성되지 못한 붕어빵 (주입 속도가 느리거나, 금형 온도가 낮을 때 주로 발생) -> Injection Speed, Mold Temp Sensor
# • 박리 (Lamination/Peeling)      : 금속 층이 서로 제대로 붙지 않고 겹쳐진 상태 (쇳물이 들어갈 때 너무 차갑거나, 불순물이 섞여 층이 생기면 발생) -> Mold Temp Sensor, Flow Rate Sensor
# • 기공 (Porosity/Gas Hole)       : 공기 방울이 송송 뚫린 상태(주조 압력이 너무 낮거나, 주입 속도가 너무 빨라서 공기가 들어갔을 때 생김) -> Casting Pressure, Injection Speed
# • 평탄 (Flatness Error)          : 바닥이 평평하지 않고 휘어진 붕어빵 (냉각 시간이 너무 짧거나, 냉각수가 골고루 잘 흐르지 못했을 때 발생) -> Cooling Time, Flow Rate Sensor
# • 개재물 (Inclusion)             : 반죽에 검은 가루나 돌멩이가 들어간 것 (재료 관리가 안되었거나, 필터에 문제가 있을 때 발생) -> Flow Rate Sensor

---
# 1.1 필요 라이브러리 및 폰트 로드

In [1]:
# ============================================================
# 라이브러리 Import
# ============================================================

# 데이터 처리 및 분석
import numpy as np
import pandas as pd
from IPython.display import display
import warnings
import platform

# 시각화
import matplotlib.pyplot as plt

# 출력 설정
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# 한글 폰트 설정
import platform
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':  # macOS
    plt.rcParams['font.family'] = 'AppleGothic'
else:  # Linux
    plt.rcParams['font.family'] = 'NanumGothic'

plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (12, 6)

# 참고: seed 고정으로 팀원 간 동일한 결과 재현 가능
np.random.seed(42)

print("="*60)
print("라이브러리 로드 완료!")
print("한글 폰트 설정 완료!")
print("="*60)


라이브러리 로드 완료!
한글 폰트 설정 완료!


# 1.2 데이터 로드

In [2]:
# 1. 원본 데이터 로드
df_original = pd.read_csv("../data/DieCasting_Quality_Raw_Data.csv", header=[0,1])

# 2. Product-1 유형의 데이터 추출
df_original_product_1 = df_original[df_original[('Process','Product_Type')] == 1].reset_index(drop=True)

# 2-1. 작업 확인용
print(f"product_type 1의 데이터 개수: {len(df_original_product_1)}개")
print('='*30)

# 2-2. csv 파일로 저장
# 인덱스는 새로운 컬럼으로 추가하지 않음
df_original_product_1.to_csv("../common-file/Data_of_Product-1.csv", index=False)


# 3. 컬럼명 중 첫번째 행을 기준으로 컬럼 분리
process_cols = [col for col in df_original_product_1.columns if col[0] == 'Process']
sensor_cols = [col for col in df_original_product_1.columns if col[0] == 'Sensor']
defects_cols = [col for col in df_original_product_1.columns if col[0] == 'Defects']

# 4. 첫 번째 컬럼을 기준으로 분리된 데이터 프레임 생성
df_process_1 = df_original_product_1[process_cols].copy()
df_sensor_1 = df_original_product_1[sensor_cols].copy()
df_defects_1 = df_original_product_1[defects_cols].copy()

# 4-1. 작업 확인용
for name, d in [('Process', df_process_1), ('Sensor', df_sensor_1), ('Defects', df_defects_1)]:
    print(f"{name}: {d.shape[1]}개 컬럼, {d.shape[0]}개")

# 5. 두 번째 행에 있는 컬럼명만 사용하도록 변경
# get_level_values(가져오고 싶은 레벨 인덱스) - level 0: 첫번째 행, level 1: 두번째 행
# 왼쪽 .columns: 컬럼명을 바꾸기 위한 설정
# 오른쪽 .columns: 기존 다중레벨 컬럼 가져오기 → get_level_values(1)로 Level 1만 추출
df_process_1.columns = df_process_1.columns.get_level_values(1)
df_sensor_1.columns = df_sensor_1.columns.get_level_values(1)
df_defects_1.columns = df_defects_1.columns.get_level_values(1)

product_type 1의 데이터 개수: 4207개
Process: 17개 컬럼, 4207개
Sensor: 14개 컬럼, 4207개
Defects: 26개 컬럼, 4207개


---
---
# 2 데이터 전처리 (Data Preprocessing)

### 2.1 중복 데이터 확인

In [3]:
# 1. 전체 행에 대한 중복값 확인 -> 완전히 동일한 행이 있는지 확인하는 용도 -> 없음
print(f"전체 행에 대한 데이터 중복값의 개수 : {df_original_product_1.duplicated().sum()}건")

# 2. ID 기준으로 중복값 확인 -> 설비 ID 중복 확인용 -> 없음
print(f"(ID 기준) 중복값의 개수 : {df_process_1.duplicated(subset=["id"]).sum()}건")


전체 행에 대한 데이터 중복값의 개수 : 0건
(ID 기준) 중복값의 개수 : 0건


---
### 2.2 데이터 타입 확인

In [4]:
# 1. 각 데이터 프레임별 컬럼의 데이터 형식 확인
print('='*30)
print("<Process 관련 데이터프레임 정보>")
print('='*30)
display(df_process_1.dtypes)

print('='*30)
print("<Sensor 관련 데이터프레임 정보>")
print('='*30)
display(df_sensor_1.dtypes)

print('='*30)
print("<Defects 관련 데이터프레임 정보>")
print('='*30)
display(df_defects_1.dtypes)

<Process 관련 데이터프레임 정보>


id                       int64
Product_Type             int64
Shot                     int64
Velocity_1             float64
Velocity_2             float64
Velocity_3             float64
High_Velocity          float64
Cylinder_Pressure        int64
Rapid_Rise_Time        float64
Biscuit_Thickness        int64
Clamping_Force           int64
Cycle_Time             float64
 Pressure_Rise_Time    float64
Casting_Pressure         int64
Spray_Time             float64
Spray_1_Time           float64
Spray_2_Time           float64
dtype: object

<Sensor 관련 데이터프레임 정보>


Melting_Furnace_Temp    float64
Air_Pressure            float64
Air_Pressure_Min          int64
Air_Pressure_Max          int64
Coolant_Temp            float64
Coolant_Temp_Min          int64
Coolant_Temp_Max          int64
Coolant_Pressure        float64
Factory_Temp            float64
Factory_Temp_Min        float64
Factory_Temp_Max        float64
Factory_Humidity        float64
Factory_Humidity_Min    float64
Factory_Humidity_Max    float64
dtype: object

<Defects 관련 데이터프레임 정보>


Short_Shot_1       int64
Bubble_1           int64
Exfoliation_1      int64
Blow_Hole_1        int64
Stain_1            int64
Dent_1             int64
Deformation_1      int64
Contamination_1    int64
Impurity_1         int64
Crack_1            int64
Scratch_1          int64
Buring_Mark_1      int64
Inclusions_1       int64
Short_Shot_2       int64
Bubble_2           int64
Exfoliation_2      int64
Blow_Hole_2        int64
Stain_2            int64
Dent_2             int64
Deformation_2      int64
Contamination_2    int64
Impurity_2         int64
Crack_2            int64
Scratch_2          int64
Buring_Mark_2      int64
Inclusions_2       int64
dtype: object

---
### 2.3 컬럼 정제

#### 2.3.1. 공백 제거 및 컬럼명 소문자 통일

In [5]:
# 1. 공백 제거 및 컬럼명 소문자 통일
df_process_1.columns = df_process_1.columns.astype("str").str.strip().str.lower()
df_sensor_1.columns = df_sensor_1.columns.astype("str").str.strip().str.lower()
df_defects_1.columns = df_defects_1.columns.astype("str").str.strip().str.lower()

In [6]:
# 2. Sensor 관련 데이터프레임에서 _min/_max로 끝나는 컬럼 찾아서 삭제
minmax_cols = [c for c in df_sensor_1.columns if c.endswith("_min") or c.endswith("_max")]
df_cleaned_sensor_1 = df_sensor_1.drop(columns=minmax_cols)


print(f"삭제한 '_min/_max'관련 컬럼 수: {len(minmax_cols)}개")
print(f"삭제 전 df_sensor_1 shape: {df_sensor_1.shape}")
print(f"삭제 후 df_sensor_1 shape: {df_cleaned_sensor_1.shape}")

삭제한 '_min/_max'관련 컬럼 수: 8개
삭제 전 df_sensor_1 shape: (4207, 14)
삭제 후 df_sensor_1 shape: (4207, 6)


#### 2.3.2 불량 유형 고유값 통일 + cavity 1,2 불량 유형 통합

In [7]:
# 1. 1(고장)이 아닌 유형의 값이 존재하는 컬럼
target_columns = [
    "short_shot_1", "exfoliation_1", "blow_hole_1", "stain_1", "deformation_1",
    "short_shot_2", "bubble_2", "exfoliation_2", "blow_hole_2", "deformation_2"
]
# 2. 1보다 큰 값을 모두 1로 변환
df_defects_1[target_columns] = df_defects_1[target_columns].clip(upper=1)

# 3. 결과 확인 <- 모든 컬럼의 고유값이 [0, 1]로 정리되었는지 확인하기 위함 
print("<1보다 큰 값은 1로 변환한 후, 고유값 확인>")
for col in df_defects_1:
    print(f"{col}: {df_defects_1[col].unique()}")

<1보다 큰 값은 1로 변환한 후, 고유값 확인>
short_shot_1: [0 1]
bubble_1: [0 1]
exfoliation_1: [0 1]
blow_hole_1: [0]
stain_1: [0 1]
dent_1: [0 1]
deformation_1: [0 1]
contamination_1: [0]
impurity_1: [0]
crack_1: [0]
scratch_1: [0]
buring_mark_1: [0]
inclusions_1: [0]
short_shot_2: [0 1]
bubble_2: [0 1]
exfoliation_2: [0 1]
blow_hole_2: [0]
stain_2: [0]
dent_2: [0]
deformation_2: [0 1]
contamination_2: [0]
impurity_2: [0]
crack_2: [0]
scratch_2: [0]
buring_mark_2: [0]
inclusions_2: [0]


In [8]:
# 4. Cavity 1,2 불량 유형 통합
df_defects_clean_step1 =df_defects_1.copy()

# 4-1. cavity 1/2 컬럼 분리
cavity_1 =df_defects_clean_step1[[c for c in df_defects_clean_step1.columns if c.endswith("_1")]].copy()
cavity_2 =df_defects_clean_step1[[c for c in df_defects_clean_step1.columns if c.endswith("_2")]].copy()

# 4-2. 컬럼명 통일: short_shot_1 -> short_shot
cavity_1.columns = [c.replace("_1", "") for c in cavity_1.columns]
cavity_2.columns = [c.replace("_2", "") for c in cavity_2.columns]

# 4-3. 제대로 분리되었는지 확인
print("cavity_1 shape:", cavity_1.shape)
print("cavity_2 shape:", cavity_2.shape)
print("cavity_1, cavity_2 컬럼이 동일한가?", set(cavity_1.columns) == set(cavity_2.columns))

cavity_1 shape: (4207, 13)
cavity_2 shape: (4207, 13)
cavity_1, cavity_2 컬럼이 동일한가? True


In [9]:
# 4-4. OR 방식으로 통합: 둘 중 하나라도 1이 있다면 1이 입력됨
defects_merged = ((cavity_1 + cavity_2) > 0).astype(int)   # (cavity_1 | cavity_2) 도 가능
df_cleaned_defects_1_step2 = defects_merged

print("통합 전 df_defects shape:",df_defects_1.shape)
print("통합 후 df_cleaned_defects_1 shape:",df_cleaned_defects_1_step2.shape)
df_cleaned_defects_1_step2.head(10)

통합 전 df_defects shape: (4207, 26)
통합 후 df_cleaned_defects_1 shape: (4207, 13)


,short_shot,bubble,exfoliation,blow_hole,stain,dent,deformation,contamination,impurity,crack,scratch,buring_mark,inclusions
0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,0,0,0
3,0,0,1,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,0,0,0
5,0,0,0,0,0,0,0,0,0,0,0,0,0
6,0,0,0,0,0,0,0,0,0,0,0,0,0
7,0,0,0,0,0,0,0,0,0,0,0,0,0
8,0,0,0,0,0,0,0,0,0,0,0,0,0
9,0,0,0,0,0,0,1,0,0,0,0,0,0


In [10]:
# 4-5. 전체 셀 기준(모든 defect 칸을 다 펼쳐서 0/1/2 비율)
flat = df_cleaned_defects_1_step2.to_numpy().ravel()
dist_all = pd.Series(flat).value_counts(normalize=True).reindex([0,1,2], fill_value=0)
print("<전체 셀 기준 0/1/2 비율>")
print((dist_all * 100).round(2).astype(str) + "%")

<전체 셀 기준 0/1/2 비율>
0    98.58%
1     1.42%
2      0.0%
Name: proportion, dtype: object


#### 2.3.3 불량 유형 분리
- 표면 불량(surface_defect) : 육안으로 확인 가능하지만, 금속의 분리나 갈라짐은 없는 불량        
    - stain(얼룩), dent(찍힘), scratch, burning_mark(소착)

- 구조 불량(structural_defect): 육안으로 금속의 분리·갈라짐이 보이거나, 제품의 강도·기능에 직접 영향을 줄 수 있는 불량 
    - short_shot(미성형), bubble(기포), blow_hole(기공), deformation(변형), crack(균열), exfoliation(박리)

- 이물질 포함 불량(contamination_defect): 원래 포함되면 안 되는 외부 물질이 들어간 불량
    - contamination(오염), impurity(이물), inclusions(개재물)

In [11]:
# 불량 유형 범주화
# 같은 유형의 불량으로 구분된 불량유형이 하나의 shot에 동시에 존재하더라도 1로 처리됨
df_defects_groups_1 = pd.DataFrame(index=df_cleaned_defects_1_step2.index)

df_defects_groups_1["surface_defect"] = ( # 표면과 관련된 불량
    df_cleaned_defects_1_step2.reindex(columns=["stain", "dent", "scratch", "burning_mark"], fill_value=0).max(axis=1)
)

df_defects_groups_1["structural_defect"] = ( # 구조와 관련된 불량
    df_cleaned_defects_1_step2.reindex(columns=["short_shot", "bubble", "blow_hole", "deformation", "crack", "exfoliation"], fill_value=0).max(axis=1)
)

df_defects_groups_1["contamination_defect"] = ( # 이물질이 포함된 불량
    df_cleaned_defects_1_step2
    .reindex(columns=["contamination", "impurity", "inclusions"], fill_value=0)
    .max(axis=1)
)


print("df_defects_groups_1 shape:", df_defects_groups_1.shape)
print('='*30)
print("<각 유형별 불량 개수>")
display(df_defects_groups_1.sum().sort_values(ascending=False)) # 각 유형별 불량 개수
print('='*30)
df_defects_groups_1.info() # Non-Null인 행의 개수 확인

df_defects_groups_1 shape: (4207, 3)
<각 유형별 불량 개수>


structural_defect       735
surface_defect            6
contamination_defect      0
dtype: int64

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4207 entries, 0 to 4206
Data columns (total 3 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   surface_defect        4207 non-null   int64
 1   structural_defect     4207 non-null   int32
 2   contamination_defect  4207 non-null   int32
dtypes: int32(2), int64(1)
memory usage: 65.9 KB


#### 2.3.4 Process 관련 데이터의 컬럼 정리
- id, shot: 새로운 식별자("shot_key")에 통합된 정보이므로 삭제
- product_type: 제품유형 1에 대한 데이터만 처리 중이므로 삭제

In [12]:
# 1. "shot_key" 컬럼 생성
df_cleaned_process_1 =df_process_1.copy()
df_cleaned_process_1["shot_key"] =df_cleaned_process_1["id"].astype(str) + "_" + df_cleaned_process_1["shot"].astype(str)

# 2. id, Shot, product_type 컬럼 삭제
df_cleaned_process_1 =df_cleaned_process_1.drop(columns=["id", "shot", "product_type"])

# 3. "shot_key" 컬럼을 맨 앞으로 이동
cols = ["shot_key"] + [c for c in df_cleaned_process_1.columns if c != "shot_key"]
df_cleaned_process_1 =df_cleaned_process_1[cols]

# 4. "shot_key"가 중복인 값이 있는지 확인
dup =df_cleaned_process_1["shot_key"][df_cleaned_process_1["shot_key"].duplicated(keep=False)]
print("중복인 값의 개수:", dup.shape[0])

중복인 값의 개수: 0


In [13]:
# 5. shot_key 컬럼이 추가된 전체 데이터프레임 제작
df_clean_product_1 = pd.concat([df_cleaned_process_1, df_cleaned_sensor_1, df_defects_groups_1], axis=1)

print("df_clean_product_1 shape:", df_clean_product_1.shape)
df_clean_product_1.head()

df_clean_product_1 shape: (4207, 24)


,shot_key,velocity_1,velocity_2,velocity_3,high_velocity,cylinder_pressure,rapid_rise_time,biscuit_thickness,clamping_force,cycle_time,pressure_rise_time,casting_pressure,spray_time,spray_1_time,spray_2_time,melting_furnace_temp,air_pressure,coolant_temp,coolant_pressure,factory_temp,factory_humidity,surface_defect,structural_defect,contamination_defect
0,1_1,0.144,0.170,0.188,2.134,214,0.008,10,258,20.7,0.044,1037,7.8,0.7,0.8,695.0,6.3,26.0,2.71,32.9,58.4,0,0,0
1,1002_2,0.144,0.170,0.182,2.124,217,0.008,11,257,20.7,0.044,1052,7.8,0.7,0.8,696.4,6.3,26.1,2.69,32.9,58.2,0,0,0
2,2003_3,0.144,0.170,0.182,2.116,214,0.008,11,257,20.8,0.041,1037,7.8,0.7,0.8,696.4,6.3,26.1,2.69,32.9,58.2,0,0,0
3,3004_4,0.144,0.170,0.182,2.137,217,0.008,11,257,20.7,0.043,1051,7.8,0.7,0.8,696.4,6.3,26.1,2.69,32.9,58.2,0,1,0
4,4005_5,0.144,0.172,0.176,2.111,217,0.008,12,257,20.7,0.042,1052,7.8,0.7,0.8,697.9,6.4,26.1,2.69,32.9,57.8,0,0,0


---
### 2.4 결측값 확인

In [14]:
# 1. 데이터프레임 내 결측값 확인 -> 없음
na_count = df_clean_product_1.isna().sum().sort_values(ascending=False)
na_cols = na_count[na_count > 0].sort_values(ascending=False)

print("<그룹별 결측치(전체 결측값의 개수)>")
print(pd.Series({
    "Process": df_cleaned_process_1.isna().sum().sum(),
    "Sensor":  df_cleaned_sensor_1.isna().sum().sum(),
    "Defects": df_defects_groups_1.isna().sum().sum()
}))
print()

print('='*30)
print("<결측치가 있는 컬럼 및 결측값의 수>")
print(na_cols.sort_values(ascending=False))
print()

<그룹별 결측치(전체 결측값의 개수)>
Process    0
Sensor     0
Defects    0
dtype: int64

<결측치가 있는 컬럼 및 결측값의 수>
Series([], dtype: int64)



#### 2.4.1 데이터셋 저장 (.csv)
- 컬럼 정제 및 불량유형 범주화 완료

In [15]:
df_clean_product_1.to_csv("../common-file/for_entire_data_product-1.csv", index=False)
df_cleaned_process_1.to_csv("../common-file/for_process_data_product-1.csv", index=False)
df_cleaned_sensor_1.to_csv("../common-file/for_sensor_data_product-1.csv", index=False)
df_defects_groups_1.to_csv("../common-file/for_defects_data_product-1.csv", index=False)